# Module 1 — Data Cleaning
## Goal: Fix all data quality issues found in EDA
### Issues to fix:
1. TotalCharges — convert string to float, drop 11 blank rows
2. SeniorCitizen — convert 0/1 to Yes/No for readability
3. Drop customerID from analysis (not a feature)
4. Drop tenure_group if it got saved (temporary EDA column)
5. Save clean dataset for all future modules

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telco_churn.csv')
print(f"Original shape: {df.shape}")

Original shape: (7043, 21)


In [2]:
# Replace blank spaces with NaN
df['TotalCharges'] = df['TotalCharges'].str.strip()
df['TotalCharges'] = df['TotalCharges'].replace('', np.nan)

# Convert to float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Confirm
print("TotalCharges dtype after fix:", df['TotalCharges'].dtype)
print("NaN values in TotalCharges:", df['TotalCharges'].isnull().sum())

TotalCharges dtype after fix: float64
NaN values in TotalCharges: 11


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
df = df.dropna(subset=['TotalCharges'])

print(f"Shape after dropping blank TotalCharges rows: {df.shape}")
print(f"Rows dropped: {7043 - len(df)}")

Shape after dropping blank TotalCharges rows: (7032, 21)
Rows dropped: 11


In [5]:
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

print("SeniorCitizen after fix:")
print(df['SeniorCitizen'].value_counts())

SeniorCitizen after fix:
SeniorCitizen
No     5890
Yes    1142
Name: count, dtype: int64


In [6]:
# Save customerID separately before dropping — we'll need it for risk scoring later
customer_ids = df['customerID'].copy()

df = df.drop(columns=['customerID'])

print(f"Shape after dropping customerID: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Shape after dropping customerID: (7032, 20)
Remaining columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [7]:
print("=== FINAL DATA QUALITY CHECK ===\n")

print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

print(f"\nMissing values:")
print(df.isnull().sum().sum(), "total missing values")

print(f"\nData types:")
print(df.dtypes)

print(f"\nChurn distribution:")
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).round(3) * 100)

=== FINAL DATA QUALITY CHECK ===

Total rows: 7032
Total columns: 20

Missing values:
0 total missing values

Data types:
gender                  str
SeniorCitizen           str
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

Churn distribution:
Churn
No     5163
Yes    1869
Name: count, dtype: int64
Churn
No     73.4
Yes    26.6
Name: proportion, dtype: float64


In [8]:
# Add customerID back before saving so we can use it for matching later
df.insert(0, 'customerID', customer_ids.values)

df.to_csv('../data/telco_churn_cleaned.csv', index=False)

print("Clean dataset saved to ../data/telco_churn_cleaned.csv")
print(f"Final shape: {df.shape}")

Clean dataset saved to ../data/telco_churn_cleaned.csv
Final shape: (7032, 21)


## Cleaning Summary

| Issue | Action | Rows Affected |
|---|---|---|
| TotalCharges stored as string | Converted to float | All rows |
| TotalCharges blank (tenure=0) | Dropped rows | 11 rows |
| SeniorCitizen stored as 0/1 | Mapped to Yes/No | All rows |
| customerID | Preserved separately for tracking | — |

**Final clean dataset: 7,032 rows × 21 columns**
**Saved as: telco_churn_cleaned.csv**